# vidcap: interactive controls

A control panel for the whole pipeline. Plug an HDMI source (of the size of the video mode you load: see the `MODE` line below) into the input and a monitor into the
output, run the two cells below, and use the widgets. Everything the widgets do is also available from Python
(see `vidcap_demo.ipynb`).

| Control | Does |
|---|---|
| **Source** | HDMI input or the built-in colour-bar test pattern |
| **Fall back** | switch to the test pattern if HDMI is missing or not the size of the loaded mode for 5 s |
| **Display** | live video, or an image from video memory (gradient, colour bars, checkerboard, your last snapshot) |
| **Filter**, **Threshold**, **Noise** | passthrough, invert, grayscale, threshold, sobel, blur, sharpen, or **noise** (retro-VCR grain; the Noise slider is its strength) |
| **Temporal average** | average the brightness of the last N frames (colour is kept); N goes up to the history reserved at start |
| **Colour grading** | pick a **Look** (teal and orange, sepia, faded film, ...), set its **Strength**, or **Upload .cube** to use your own table; a preview shows what the look does to a test picture |
| **Timestamp** | show/hide, size, margin, colours, transparent box, and set the clock from this board |
| **Capture snapshot** | grabs the next full frame from the selected input (before filtering); **Save as PNG** keeps it |
| **Status** | HDMI state, source in use, display, filter, colour grading, hardware clock and drift, DMA errors |

Errors from any control appear under **Messages** at the bottom.

**One notebook at a time.** Both notebooks load the bitstream, and only one process should drive the hardware. Before opening the other notebook, close this one (File → Close and Shut Down Notebook, or Kernel → Shut Down All Kernels). If the picture freezes or snapshots never finish, run `sudo python3 -m vidcap.diagnose` from this folder: it reads the hardware without changing it and says where the video stopped.

In [ ]:
import sys
sys.path.insert(0, '.')            # the 'vidcap' package sits next to this notebook
from vidcap import Vidcap, ui

MODE = '720p'                      # '720p', '480p', '240p', '1080p30' or 'xga': each mode is its own bitstream
v = Vidcap.on_pynq(mode=MODE)      # loads the bitstream and checks that every block answers
v.start()                          # starts the DMA engines and shows live video

In [ ]:
panel = ui.build_panel(v)
display(panel)

## Notes

* The **test pattern** has a box that moves every frame. If it stops, the pipeline has stalled.
* **Colour grading** is applied after the filter and before the timestamp, so the clock text keeps its colours. An uploaded .cube file is checked on the board after the load (write count and checksum); a file that cannot be read is reported under Messages.
* **Video memory** images go through the filter and get the timestamp, just like live video.
* Saved snapshots go to `snapshots/` next to this notebook.
* A red **DMA errors** line in the status panel means a VDMA reported a stream error; the names say which.

In [ ]:
# run this before shutting the kernel down, to end the status refresh
panel.stop()